# 官方公司行動與上市主檔（B3／B5）：探索、試抓、正式抓取

抓取證交所與櫃買的公開報表，存成新的版本化參考資料 `reference/corporate_actions/v1`：
- 除權息
- 減資
- ETF 分割
- 暫停交易
- 上下市主檔

**為什麼分三步**：交易所網頁報表的名稱和參數會改。開發環境連不到交易所網站，所以程式裡的端點都只是候選。

1. **第 1–6 格（探索與試抓，2026-10-30 已完成）**：結果在 `audits/corporate_actions_probe/`。依據這次結果，程式已選定 `REPORTS = "recommended"`，共 25 個報表。之後不必再跑第 4–6 格。
2. **第 1–3 格，再跑第 7–9 格（正式抓取，可分段）**：
   - 從 2023-05 起，依報表別逐月、逐年或逐日抓取。
   - 已完成的期間會跳過，到 `MAX_MINUTES` 會自動停止，下次重跑同樣幾格就會接著抓。
   - 某個報表連續失敗 3 次，這次就跳過它，其他報表照常繼續；失敗的報表會列在輸出最後。
   - 證交所約 15 分鐘。櫃買約 50–60 分鐘，其中逐日的「變更交易、停止交易資訊」排在最後。
3. **第 10 格（解析，無網路）**：把保存的表格攤平成長表，不做任何欄位解讀。

**寫入範圍**：
- 只寫到 `_tw-market-data-lake/reference/corporate_actions/v1/` 與 `audits/corporate_actions_probe/`。
- 不碰任何 lake 資料集、manifest、pointer、遮罩，也不碰 `official_volume`。
- 只呼叫公開報表，不使用任何帳號或下單 API。


In [ ]:
# 1. 參數
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
DATASET_ROOT = f"{DATA_ROOT}/_tw-market-data-lake/reference/corporate_actions/v1"
PROBE_ARCHIVE = f"{DATA_ROOT}/_tw-market-data-lake/audits/corporate_actions_probe"
PROJECT_DIR = f"/content/{REPO_NAME}"
OUT_DIR = "/content/tw_lake_ca"           # 本機暫存
PROBE_START, PROBE_END = "2025-06-01", "2025-06-30"
REPORTS = "recommended"                  # 依 2026-10-30 試抓結果選定；`list` 可看全部候選
FETCH_START = "2023-05-01"
MAX_MINUTES = 55


In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. clone / pull + 依賴
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)

In [ ]:
# 4. 探索：列出交易所網站實際使用的 API（約 2–4 分鐘）
print("exit", stream([sys.executable, "-m", "tw_lake.reference.corporate_actions", "discover", "--out-dir", OUT_DIR]))


In [ ]:
# 5. 試抓：每個候選端點抓 PROBE_START–PROBE_END 一次（約 1–2 分鐘）
print("exit", stream([sys.executable, "-m", "tw_lake.reference.corporate_actions", "probe", "--out-dir", OUT_DIR,
                      "--start", PROBE_START, "--end", PROBE_END]))


In [ ]:
# 6. 顯示結果並複製到 Drive（把這格的輸出貼給 Claude）
import glob, shutil, os
import pandas as pd
ensure_drive()
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 120); pd.set_option("display.max_rows", 200)
probe_dir = sorted(glob.glob(f"{OUT_DIR}/ca_probe_*"))[-1]
disc_dir = sorted(glob.glob(f"{OUT_DIR}/ca_discovery_*"))[-1]
s = pd.read_csv(f"{probe_dir}/probe_summary.csv")
print(s[["report", "http_status", "json", "html", "stat", "tables", "rows", "fields"]].to_string(index=False))
print("\nfirst rows:")
for r in s[s["rows"] > 0].itertuples():
    print(r.report, "|", str(r.first_row)[:300])
d = pd.read_csv(f"{disc_dir}/discovered_actions.csv").fillna("")
print("\ndiscovered actions:")
print(d[d["actions"] != ""][["site", "title", "url", "http_status", "actions"]].to_string(index=False))
for src in (probe_dir, disc_dir):
    dest = f"{PROBE_ARCHIVE}/{os.path.basename(src)}"
    shutil.copytree(src, dest)
    print("copied to", dest)


In [ ]:
# 7. 正式抓取（到 MAX_MINUTES 自動停止；可隨時中斷，已完成的期間不會重抓；最後會印出各報表進度）
ensure_drive()
print("exit", stream([sys.executable, "-m", "tw_lake.reference.corporate_actions", "fetch", "--dataset-root", DATASET_ROOT,
                      "--work-dir", f"{OUT_DIR}/work", "--reports", REPORTS, "--start", FETCH_START,
                      "--max-minutes", str(MAX_MINUTES)]))


In [ ]:
# 8. 進度
print("exit", stream([sys.executable, "-m", "tw_lake.reference.corporate_actions", "status", "--dataset-root", DATASET_ROOT,
                      "--reports", REPORTS, "--start", FETCH_START]))


In [ ]:
# 9. 確保上傳到 Drive（每次抓完都要跑）
from google.colab import drive
drive.flush_and_unmount()
print("flushed; 下次重跑第 1–3、7–9 格即可接著抓")


In [ ]:
# 10. 解析（無網路；全部抓完後再跑），結果複製到 Drive
import glob, os, shutil
import pandas as pd
ensure_drive()
print("exit", stream([sys.executable, "-m", "tw_lake.reference.corporate_actions", "parse", "--dataset-root", DATASET_ROOT,
                      "--out-dir", f"{OUT_DIR}/parsed"]))
pdir = sorted(glob.glob(f"{OUT_DIR}/parsed/ca_parsed_*"))[-1]
print(pd.read_csv(f"{pdir}/tables_summary.csv").to_string(index=False))
dest = f"{DATA_ROOT}/_tw-market-data-lake/audits/corporate_actions_parsed/{os.path.basename(pdir)}"
shutil.copytree(pdir, dest)
print("copied to", dest)
from google.colab import drive
drive.flush_and_unmount()
